# 06. Проверка файла на антиплагиат

In [1]:
import pickle
import re
from pathlib import Path

import fitz 
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import nltk
from nltk.corpus import stopwords
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

nltk.download('stopwords', quiet=True)

OUT_DIR = Path('artifacts')

MODEL_ID           = 'ibm-granite/granite-embedding-311m-multilingual-r2'
MAX_LEN            = 512
OVERLAP            = 64
BATCH_SIZE         = 16
MATCH_THRESHOLD    = 0.90
SUSPECT_OVERLAP    = 0.20
SKIP_INTRO_CHUNKS  = 2
TOP_K_MATCHES      = 10

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

device: cuda


In [2]:
embeddings  = np.load(OUT_DIR / 'embeddings.npy')
chunks_meta = pd.read_pickle(OUT_DIR / 'chunks_meta.pkl')
df_corpus   = pd.read_pickle(OUT_DIR / 'df_corpus_ready.pkl').reset_index(drop=True)
with open(OUT_DIR / 'embeddings_config.pkl', 'rb') as f:
    cfg = pickle.load(f)

print(f'Корпус       : {len(df_corpus)} документов, {len(chunks_meta)} чанков')
print(f'Эмбеддинги   : {embeddings.shape}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model     = AutoModel.from_pretrained(MODEL_ID).to(device).eval()
print(f'Модель       : {MODEL_ID} загружена')

Корпус       : 194 документов, 4986 чанков
Эмбеддинги   : (4986, 768)


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Модель       : ibm-granite/granite-embedding-311m-multilingual-r2 загружена


In [3]:
STOP = (
    set(stopwords.words('russian')) |
    set(stopwords.words('english')) |
    set(stopwords.words('kazakh'))
)

BOILER_PATS = [
    r'қазақстан\s+республикасы',
    r'министрлігі|министрлiгi',
    r'сәтбаев\s+университет|satbayev\s+university',
    r'бекітемін|бекiтемiн',
    r'қорғауға\s+жіберілді',
    r'ғылыми\s+жетекшінің\s+сын',
    r'программалық\s+инженерия\s+кафедрасы',
    r'отчет\s+подобия',
    r'scanned\s+with',
]
ABSTRACT_PATS = [r'аңдатпа', r'аннотация', r'\bannotation\b', r'\babstract\b']
TOC_PATS      = [r'мазмұны', r'содержание', r'table\s+of\s+contents']
BIBLIO_PATS   = [
    r'пайдаланылған\s+әдебиеттер',
    r'список\s+(использованн|литератур)',
    r'\bbibliography\b',
    r'\breferences\b',
]
APPENDIX_PATS = [
    r'(?:^|\n)\s*қосымша(?:лар)?\s*[а-дА-ДA-D1-9]',
    r'(?:^|\n)\s*қосымшалар\s*$',
    r'(?:^|\n)\s*приложени[ея]',
    r'(?:^|\n)\s*appendix\b',
]

MIN_CHARS  = 30
HEADER_LEN = 500


def extract_text_from_pdf(pdf_path: str) -> list[str]:
    doc = fitz.open(pdf_path)
    pages = []
    for page in doc:
        pages.append(page.get_text().strip())
    doc.close()
    return pages


def extract_body(pages: list[str]) -> str:
    n = len(pages)
    if n == 0:
        return ''

    end_idx = n
    search_from = max(n // 2, 5)
    for i in range(n - 1, search_from - 1, -1):
        header = pages[i].lower().strip()[:HEADER_LEN]
        if len(header) < MIN_CHARS:
            continue
        if any(re.search(p, header) for p in BIBLIO_PATS):
            end_idx = i
            break
        if any(re.search(p, header, re.MULTILINE) for p in APPENDIX_PATS):
            end_idx = i
            break

    max_front_skip = min(10, n // 3 + 1)
    skip_set = set()
    for i in range(min(max_front_skip, end_idx)):
        text = pages[i].strip()
        if len(text) < MIN_CHARS:
            skip_set.add(i)
            continue
        header = text.lower()[:HEADER_LEN]
        if any(re.search(p, header) for p in BOILER_PATS + ABSTRACT_PATS + TOC_PATS):
            skip_set.add(i)

    body_parts = []
    for i in range(end_idx):
        if i in skip_set:
            continue
        text = pages[i].strip()
        if len(text) < MIN_CHARS:
            continue
        body_parts.append(text)

    return '\n'.join(body_parts)


def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r'[^0-9a-zа-яәіңғүұқөһё\s]', ' ', text)
    tokens = [w for w in text.split() if w not in STOP and len(w) > 1]
    return re.sub(r'\s+', ' ', ' '.join(tokens)).strip()

In [4]:
def chunk_document(text: str, tokenizer, max_len: int = MAX_LEN, overlap: int = OVERLAP):
    if not text or not text.strip():
        return []
    ids = tokenizer.encode(text, add_special_tokens=False)
    if not ids:
        return []
    stride = max_len - overlap
    window = max_len - 2
    step   = stride
    chunks = []
    for start in range(0, len(ids), step):
        window_ids = ids[start:start + window]
        if len(window_ids) < 16:
            break
        chunks.append({'token_start': start, 'token_end': start + len(window_ids), 'ids': window_ids})
        if start + window >= len(ids):
            break
    return chunks


_CLS_ID = tokenizer.cls_token_id
_SEP_ID = tokenizer.sep_token_id
_PAD_ID = tokenizer.pad_token_id
if _PAD_ID is None:
    _PAD_ID = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0


def mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).float()
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


def build_batch(chunk_ids_list, max_len=MAX_LEN):
    batch_ids = []
    for ids in chunk_ids_list:
        full = list(ids)
        if _CLS_ID is not None:
            full = [_CLS_ID] + full
        if _SEP_ID is not None:
            full = full + [_SEP_ID]
        full = full[:max_len]
        batch_ids.append(full)
    L = max(len(x) for x in batch_ids)
    input_ids = torch.full((len(batch_ids), L), _PAD_ID, dtype=torch.long)
    attention_mask = torch.zeros((len(batch_ids), L), dtype=torch.long)
    for i, ids in enumerate(batch_ids):
        n_tok = len(ids)
        input_ids[i, :n_tok] = torch.tensor(ids, dtype=torch.long)
        attention_mask[i, :n_tok] = 1
    return input_ids, attention_mask


@torch.no_grad()
def embed_chunks_list(chunks: list[dict], model, batch_size=BATCH_SIZE, device=device):
    """Генерация эмбеддингов для списка чанков."""
    n = len(chunks)
    hidden = model.config.hidden_size
    out = np.zeros((n, hidden), dtype=np.float32)
    use_amp = device.type == 'cuda'
    ids_list = [c['ids'] for c in chunks]

    for start in tqdm(range(0, n, batch_size), desc='Embedding query'):
        end = min(start + batch_size, n)
        input_ids, attention_mask = build_batch(ids_list[start:end])
        input_ids      = input_ids.to(device, non_blocking=True)
        attention_mask = attention_mask.to(device, non_blocking=True)

        if use_amp:
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                output = model(input_ids=input_ids, attention_mask=attention_mask)
                pooled = mean_pool(output.last_hidden_state, attention_mask)
                pooled = F.normalize(pooled, p=2, dim=1)
        else:
            output = model(input_ids=input_ids, attention_mask=attention_mask)
            pooled = mean_pool(output.last_hidden_state, attention_mask)
            pooled = F.normalize(pooled, p=2, dim=1)

        out[start:end] = pooled.float().cpu().numpy()
    return out

In [5]:
def check_plagiarism(pdf_path: str,
                     match_threshold: float = MATCH_THRESHOLD,
                     suspect_overlap: float = SUSPECT_OVERLAP,
                     skip_intro: int = SKIP_INTRO_CHUNKS,
                     top_k: int = TOP_K_MATCHES):

    pdf_path = Path(pdf_path)
    assert pdf_path.exists(), f'Файл не найден: {pdf_path}'
    print(f'\n{"=" * 80}')
    print(f'Файл: {pdf_path.name}')
    print(f'{"=" * 80}')

    pages = extract_text_from_pdf(str(pdf_path))
    print(f'\n[1/5] Извлечено страниц: {len(pages)}')

    body = extract_body(pages)
    if not body.strip():
        print('⚠ Не удалось извлечь текст из PDF.')
        return None
    print(f'[2/5] Тело документа: {len(body)} символов, ~{len(body.split())} слов')

    cleaned = clean_text(body)
    wc = len(cleaned.split())
    print(f'[3/5] После очистки: {wc} слов')
    if wc < 100:
        print('⚠ Слишком мало текста после очистки.')
        return None

    doc_chunks = chunk_document(cleaned, tokenizer)
    print(f'[4/5] Чанков: {len(doc_chunks)}')
    if len(doc_chunks) < skip_intro + 1:
        print('⚠ Недостаточно чанков для анализа.')
        return None

    query_emb = embed_chunks_list(doc_chunks, model)
    print(f'      Эмбеддинги: {query_emb.shape}')

    print(f'[5/5] Сравнение с корпусом ({len(df_corpus)} документов)...')

    query_idx = list(range(skip_intro, len(doc_chunks)))
    if not query_idx:
        query_idx = list(range(len(doc_chunks)))
    q_emb = query_emb[query_idx]

    sim = q_emb @ embeddings.T

    results = []
    for doc_id in df_corpus.index:
        corpus_idx = chunks_meta.index[
            (chunks_meta['doc_idx'] == doc_id) &
            (chunks_meta['chunk_idx'] >= skip_intro)
        ].to_numpy()
        if len(corpus_idx) < 3:
            continue

        sub = sim[:, corpus_idx]
        best_q = sub.max(axis=1)
        best_c = sub.max(axis=0)

        matched_q = int((best_q > match_threshold).sum())
        matched_c = int((best_c > match_threshold).sum())
        overlap_q = matched_q / len(query_idx)
        overlap_c = matched_c / len(corpus_idx)
        score = max(overlap_q, overlap_c)
        max_sim = float(sub.max())

        if matched_q >= 1 or matched_c >= 1:
            flat = sub.flatten()
            k = min(top_k, len(flat))
            top_idx = np.argpartition(flat, -k)[-k:]
            top_idx = top_idx[np.argsort(flat[top_idx])[::-1]]
            matches = []
            for idx in top_idx:
                qi = int(idx // sub.shape[1])
                ci = int(idx % sub.shape[1])
                matches.append({
                    'query_chunk': query_idx[qi],
                    'corpus_chunk_global': int(corpus_idx[ci]),
                    'corpus_chunk_idx': int(chunks_meta.iloc[corpus_idx[ci]]['chunk_idx']),
                    'sim': float(flat[idx]),
                })

            results.append({
                'doc_idx'    : doc_id,
                'filename'   : df_corpus.loc[doc_id, 'filename'],
                'year'       : df_corpus.loc[doc_id, 'year'],
                'matched_q'  : matched_q,
                'matched_c'  : matched_c,
                'n_chunks_q' : len(query_idx),
                'n_chunks_c' : len(corpus_idx),
                'overlap_q'  : overlap_q,
                'overlap_c'  : overlap_c,
                'score'      : score,
                'max_sim'    : max_sim,
                'top_matches': matches,
            })

    results.sort(key=lambda x: x['score'], reverse=True)

    suspect = [r for r in results if r['score'] >= suspect_overlap]
    print(f'\n{"─" * 80}')
    print(f'РЕЗУЛЬТАТ: найдено {len(suspect)} подозрительных совпадений '
          f'(score >= {suspect_overlap}) из {len(results)} пар с совпадениями')
    print(f'{"─" * 80}')

    if suspect:
        print(f'\n{"🔴 ОБНАРУЖЕН ПЛАГИАТ" if suspect[0]["score"] > 0.5 else "🟡 ПОДОЗРИТЕЛЬНЫЕ СОВПАДЕНИЯ"}')
        for i, r in enumerate(suspect[:10]):
            print(f'\n  [{i+1}] {r["filename"][:70]}  ({r["year"]})')
            print(f'      Score: {r["score"]:.1%}  |  '
                  f'Совпало: {r["matched_q"]}/{r["n_chunks_q"]} (запрос), '
                  f'{r["matched_c"]}/{r["n_chunks_c"]} (корпус)  |  '
                  f'Max sim: {r["max_sim"]:.3f}')
            print(f'      Топ совпадения:')
            for m in r['top_matches'][:5]:
                print(f'        chunk Q[{m["query_chunk"]:2d}] ↔ C[{m["corpus_chunk_idx"]:2d}]  sim={m["sim"]:.3f}')
    else:
        print('\n🟢 Плагиат не обнаружен.')
        if results:
            top3 = results[:3]
            print(f'\nБлижайшие документы:')
            for r in top3:
                print(f'  {r["filename"][:70]}  score={r["score"]:.1%}  max_sim={r["max_sim"]:.3f}')

    return {
        'filename': pdf_path.name,
        'n_pages': len(pages),
        'n_words_clean': wc,
        'n_chunks': len(doc_chunks),
        'suspect_count': len(suspect),
        'all_results': results,
    }

In [11]:
PDF_PATH = 'C:/Users/bauir/OneDrive/Рабочий стол/PREFINAL DIPLOMA.pdf'

result = check_plagiarism(PDF_PATH)


Файл: PREFINAL DIPLOMA.pdf

[1/5] Извлечено страниц: 54
[2/5] Тело документа: 74613 символов, ~8091 слов
[3/5] После очистки: 7584 слов
[4/5] Чанков: 50


Embedding query:   0%|          | 0/4 [00:00<?, ?it/s]

      Эмбеддинги: (50, 768)
[5/5] Сравнение с корпусом (194 документов)...

────────────────────────────────────────────────────────────────────────────────
РЕЗУЛЬТАТ: найдено 0 подозрительных совпадений (score >= 0.2) из 0 пар с совпадениями
────────────────────────────────────────────────────────────────────────────────

🟢 Плагиат не обнаружен.


In [12]:
PDF_PATH = 'data_PI/2025/БАК 2025 3 Абдуали Дінислам, Рақыметқан Ләзат.pdf'

result = check_plagiarism(PDF_PATH)


Файл: БАК 2025 3 Абдуали Дінислам, Рақыметқан Ләзат.pdf

[1/5] Извлечено страниц: 73
[2/5] Тело документа: 83881 символов, ~9666 слов
[3/5] После очистки: 8515 слов
[4/5] Чанков: 58


Embedding query:   0%|          | 0/4 [00:00<?, ?it/s]

      Эмбеддинги: (58, 768)
[5/5] Сравнение с корпусом (194 документов)...

────────────────────────────────────────────────────────────────────────────────
РЕЗУЛЬТАТ: найдено 1 подозрительных совпадений (score >= 0.2) из 10 пар с совпадениями
────────────────────────────────────────────────────────────────────────────────

🔴 ОБНАРУЖЕН ПЛАГИАТ

  [1] БАК 2025 3 Абдуали Дінислам, Рақыметқан Ләзат.pdf  (2025)
      Score: 100.0%  |  Совпало: 56/56 (запрос), 56/56 (корпус)  |  Max sim: 1.000
      Топ совпадения:
        chunk Q[ 9] ↔ C[ 9]  sim=1.000
        chunk Q[24] ↔ C[24]  sim=1.000
        chunk Q[20] ↔ C[20]  sim=1.000
        chunk Q[10] ↔ C[10]  sim=1.000
        chunk Q[12] ↔ C[12]  sim=1.000


In [13]:
PDF_PATH = 'data_PI/2025/БАК 2025 5 Абдулла К, Араова А.pdf'

result = check_plagiarism(PDF_PATH)


Файл: БАК 2025 5 Абдулла К, Араова А.pdf

[1/5] Извлечено страниц: 69
[2/5] Тело документа: 75253 символов, ~8815 слов
[3/5] После очистки: 7008 слов
[4/5] Чанков: 26


Embedding query:   0%|          | 0/2 [00:00<?, ?it/s]

      Эмбеддинги: (26, 768)
[5/5] Сравнение с корпусом (194 документов)...

────────────────────────────────────────────────────────────────────────────────
РЕЗУЛЬТАТ: найдено 1 подозрительных совпадений (score >= 0.2) из 2 пар с совпадениями
────────────────────────────────────────────────────────────────────────────────

🔴 ОБНАРУЖЕН ПЛАГИАТ

  [1] БАК 2025 5 Абдулла К, Араова А.pdf  (2025)
      Score: 100.0%  |  Совпало: 24/24 (запрос), 24/24 (корпус)  |  Max sim: 1.000
      Топ совпадения:
        chunk Q[25] ↔ C[25]  sim=1.000
        chunk Q[15] ↔ C[15]  sim=1.000
        chunk Q[24] ↔ C[24]  sim=1.000
        chunk Q[11] ↔ C[11]  sim=1.000
        chunk Q[10] ↔ C[10]  sim=1.000
